In [ ]:
#@title Setup — Run this cell first
import warnings
warnings.filterwarnings('ignore')

# ── Embedded company_sim.py ──
"""
company_sim.py — Shared infrastructure for the Regression Autopsy course.
Embedded in every notebook's setup cell. Self-contained, no external deps
beyond numpy, pandas, matplotlib, statsmodels, ipywidgets.
"""

import json
import os
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence

import ipywidgets as widgets

# ---------------------------------------------------------------------------
# Color constants — consistent across all notebooks
# ---------------------------------------------------------------------------
COLORS = {
    'ols': '#2E5EA8',       # Blue — OLS estimator
    'truth': '#D4A017',     # Gold — true parameter
    'bias': '#C0392B',      # Red — bias/violation/error
    'repair': '#27AE60',    # Green — repair working
    'alt': '#7F8C8D',       # Gray — alternative estimators
}

# ---------------------------------------------------------------------------
# Gate System — module-level trap response storage
# ---------------------------------------------------------------------------
_trap_responses = {}
_TRAP_LOG_PATH = '/content/trap_log.json'


def _load_trap_log():
    """Load existing trap log from disk into memory."""
    global _trap_responses
    if os.path.exists(_TRAP_LOG_PATH):
        try:
            with open(_TRAP_LOG_PATH, 'r') as f:
                _trap_responses = json.load(f)
        except (json.JSONDecodeError, IOError):
            _trap_responses = {}


def _save_trap_log():
    """Persist in-memory trap responses to disk."""
    try:
        os.makedirs(os.path.dirname(_TRAP_LOG_PATH), exist_ok=True)
        with open(_TRAP_LOG_PATH, 'w') as f:
            json.dump(_trap_responses, f, indent=2)
    except IOError:
        pass  # Silently fail if /content not available (local dev)


def record_trap_response(notebook_id, question_id, response):
    """Save a trap response to the log."""
    key = f"{notebook_id}_{question_id}"
    _trap_responses[key] = {
        "response": response,
        "timestamp": datetime.now().isoformat(),
    }
    _save_trap_log()


def get_trap_response(notebook_id, question_id):
    """Retrieve a previously recorded trap response, or None."""
    key = f"{notebook_id}_{question_id}"
    entry = _trap_responses.get(key)
    return entry["response"] if entry else None


def check_gate(notebook_id, question_id):
    """Return True if a response has been recorded for this gate."""
    key = f"{notebook_id}_{question_id}"
    return key in _trap_responses


def get_all_responses():
    """Return all recorded trap responses."""
    return dict(_trap_responses)


def create_trap_widget(notebook_id, question_id, question_text, options):
    """Create a radio-button trap widget with submit button and gate logic."""
    label = widgets.HTML(f"<b>{question_text}</b>")
    radio = widgets.RadioButtons(
        options=options,
        value=None,
        layout=widgets.Layout(width='auto'),
    )
    submit = widgets.Button(description="Submit", button_style='primary')
    output = widgets.Output()

    def on_submit(btn):
        with output:
            output.clear_output()
            if radio.value is None:
                print("Please select an option before submitting.")
                return
            record_trap_response(notebook_id, question_id, radio.value)
            print(f"Response recorded: {radio.value}")
            submit.disabled = True
            radio.disabled = True

    submit.on_click(on_submit)

    # Pre-fill if already answered
    existing = get_trap_response(notebook_id, question_id)
    if existing is not None:
        try:
            radio.value = existing
            radio.disabled = True
            submit.disabled = True
        except Exception:
            pass

    return widgets.VBox([label, radio, submit, output])


# Load any existing responses at import time
_load_trap_log()


# ---------------------------------------------------------------------------
# CompanySimulator — NovaMart data generating process
# ---------------------------------------------------------------------------
class CompanySimulator:
    """
    Generates simulated NovaMart data with controllable pathologies:
    omitted variable bias, heteroscedasticity, nonlinearity, bad controls.
    """

    def __init__(
        self,
        endogeneity_strength=20,
        heteroscedasticity_strength=0.5,
        nonlinearity=True,
        bad_control_strength=0.1,
        noise_sigma=1.0,
    ):
        self.endogeneity_strength = endogeneity_strength
        self.heteroscedasticity_strength = heteroscedasticity_strength
        self.nonlinearity = nonlinearity
        self.bad_control_strength = bad_control_strength
        self.noise_sigma = noise_sigma

        # True DGP parameters
        self.beta_0 = 50
        self.beta_1 = 8      # coefficient on log(X1) or X1
        self.beta_2 = 3      # coefficient on X2
        self.beta_U = 2      # coefficient on U
        self.staff_loading = 5  # U -> X2 coefficient

    def generate(self, n=500, seed=42):
        """Generate observed data: revenue, ad_spend, staff_count, satisfaction."""
        rng = np.random.default_rng(seed)

        # Step 1: Generate U, noise terms, epsilon base
        U = rng.standard_normal(n)
        eta1 = rng.normal(0, self.noise_sigma, n)
        eta2 = rng.normal(0, self.noise_sigma, n)
        eta3 = rng.normal(0, self.noise_sigma, n)

        # Step 2: X1, X2 from U
        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity:
            X1 = np.abs(X1) + 0.01  # Ensure positive for log
        X2 = self.staff_loading * U + eta2

        # Step 3: Compute Y
        # Heteroscedastic errors: eps ~ N(0, h * X1)
        eps_std = np.sqrt(np.abs(self.heteroscedasticity_strength * X1))
        eps = rng.normal(0, eps_std)

        if self.nonlinearity:
            Y = self.beta_0 + self.beta_1 * np.log(X1) + self.beta_2 * X2 + self.beta_U * U + eps
        else:
            Y = self.beta_0 + self.beta_1 * X1 + self.beta_2 * X2 + self.beta_U * U + eps

        # Step 4: X3 from Y (post-treatment / bad control)
        X3 = self.bad_control_strength * Y + eta3

        return pd.DataFrame({
            'revenue': Y,
            'ad_spend': X1,
            'staff_count': X2,
            'satisfaction': X3,
        })

    def truth(self, n=500, seed=42):
        """Generate data including hidden demand_U + return true parameter dict."""
        rng = np.random.default_rng(seed)

        U = rng.standard_normal(n)
        eta1 = rng.normal(0, self.noise_sigma, n)
        eta2 = rng.normal(0, self.noise_sigma, n)
        eta3 = rng.normal(0, self.noise_sigma, n)

        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity:
            X1 = np.abs(X1) + 0.01
        X2 = self.staff_loading * U + eta2

        eps_std = np.sqrt(np.abs(self.heteroscedasticity_strength * X1))
        eps = rng.normal(0, eps_std)

        if self.nonlinearity:
            Y = self.beta_0 + self.beta_1 * np.log(X1) + self.beta_2 * X2 + self.beta_U * U + eps
        else:
            Y = self.beta_0 + self.beta_1 * X1 + self.beta_2 * X2 + self.beta_U * U + eps

        X3 = self.bad_control_strength * Y + eta3

        df = pd.DataFrame({
            'revenue': Y,
            'ad_spend': X1,
            'staff_count': X2,
            'satisfaction': X3,
            'demand_U': U,
        })

        params = {
            'beta_0': self.beta_0,
            'beta_1': self.beta_1,
            'beta_2': self.beta_2,
            'beta_U': self.beta_U,
            'sigma_epsilon': self.heteroscedasticity_strength,
        }

        return df, params

    def dgp_summary(self):
        """Return LaTeX specification string."""
        func = r"\log(X_1)" if self.nonlinearity else r"X_1"
        return (
            rf"$Y = {self.beta_0} + {self.beta_1} \cdot {func} "
            rf"+ {self.beta_2} \cdot X_2 + {self.beta_U} \cdot U + \varepsilon$"
            "\n\nWhere:\n"
            rf"- $U \sim N(0, 1)$ (unobserved market demand)"
            "\n"
            rf"- $X_1 = {self.endogeneity_strength} \cdot U + \eta_1$ (ad spend)"
            "\n"
            rf"- $X_2 = {self.staff_loading} \cdot U + \eta_2$ (staff count)"
            "\n"
            rf"- $X_3 = {self.bad_control_strength} \cdot Y + \eta_3$ (satisfaction, post-treatment)"
            "\n"
            rf"- $\varepsilon \sim N(0, {self.heteroscedasticity_strength} \cdot X_1)$"
            "\n"
            rf"- $\eta_i \sim N(0, {self.noise_sigma}^2)$"
        )

    def set_heteroscedasticity(self, strength):
        """Update heteroscedasticity strength."""
        self.heteroscedasticity_strength = strength

    def set_endogeneity(self, strength):
        """Update endogeneity strength (U -> X1 coefficient)."""
        self.endogeneity_strength = strength

    def set_nonlinearity(self, curvature):
        """Toggle or adjust nonlinearity. Pass bool or truthy value."""
        self.nonlinearity = bool(curvature)


# ---------------------------------------------------------------------------
# MonteCarloEngine — precompute simulation results for interactive sliders
# ---------------------------------------------------------------------------
class MonteCarloEngine:
    """Precomputes Monte Carlo draws across a parameter grid."""

    def run(self, estimator_fn, param_name, param_grid, simulator,
            n_reps=5000, n_obs=500):
        """
        For each value in param_grid, set param on simulator, run n_reps
        replications, collect estimator_fn output.

        Returns numpy array of shape (len(param_grid), n_reps).
        """
        results = np.empty((len(param_grid), n_reps))

        # Progress bar
        try:
            progress = widgets.IntProgress(
                value=0, min=0, max=len(param_grid),
                description='Simulating:', style={'description_width': 'initial'},
            )
            from IPython.display import display
            display(progress)
            use_widget = True
        except Exception:
            use_widget = False

        setter = getattr(simulator, f'set_{param_name}', None)

        for i, val in enumerate(param_grid):
            if setter is not None:
                setter(val)
            for r in range(n_reps):
                data = simulator.generate(n=n_obs, seed=r)
                results[i, r] = estimator_fn(data)
            if use_widget:
                progress.value = i + 1
            else:
                print(f"  [{i+1}/{len(param_grid)}] {param_name}={val:.3f}")

        if use_widget:
            progress.bar_style = 'success'

        return results

    def quick_run(self, estimator_fn, dgp_fn, n_reps=1000, n_obs=500):
        """
        Single-configuration simulation for sidebar mini-sims.
        dgp_fn(seed, n) -> DataFrame. No caching.
        """
        results = np.empty(n_reps)
        for r in range(n_reps):
            data = dgp_fn(seed=r, n=n_obs)
            results[r] = estimator_fn(data)
        return results


# ---------------------------------------------------------------------------
# DiagnosticSuite — standardized diagnostic visualizations
# ---------------------------------------------------------------------------
class DiagnosticSuite:
    """Produces four-panel residual diagnostics and summary test statistics."""

    @staticmethod
    def run_diagnostics(model_result):
        """
        2x2 diagnostic plot grid from a statsmodels OLS results object.
        Returns the matplotlib Figure.
        """
        fig, axes = plt.subplots(2, 2, figsize=(10, 8))

        influence = OLSInfluence(model_result)
        fitted = model_result.fittedvalues
        resid = model_result.resid
        std_resid = influence.resid_studentized_internal

        # Top-left: Residuals vs Fitted
        ax = axes[0, 0]
        ax.scatter(fitted, resid, alpha=0.4, s=15, color=COLORS['ols'])
        ax.axhline(0, color=COLORS['truth'], linewidth=1.5)
        ax.set_xlabel('Fitted values')
        ax.set_ylabel('Residuals')
        ax.set_title('Residuals vs Fitted')

        # Top-right: QQ plot
        ax = axes[0, 1]
        stats.probplot(resid, dist="norm", plot=ax)
        ax.set_title('Normal Q-Q')
        ax.get_lines()[0].set_color(COLORS['ols'])
        ax.get_lines()[1].set_color(COLORS['bias'])

        # Bottom-left: Scale-Location
        ax = axes[1, 0]
        sqrt_abs_std = np.sqrt(np.abs(std_resid))
        ax.scatter(fitted, sqrt_abs_std, alpha=0.4, s=15, color=COLORS['ols'])
        ax.set_xlabel('Fitted values')
        ax.set_ylabel(r'$\sqrt{|Standardized\ Residuals|}$')
        ax.set_title('Scale-Location')

        # Bottom-right: Residuals vs Leverage
        ax = axes[1, 1]
        leverage = influence.hat_matrix_diag
        cooks_d = influence.cooks_distance[0]
        ax.scatter(leverage, std_resid, alpha=0.4, s=15, color=COLORS['ols'])
        ax.axhline(0, color=COLORS['truth'], linewidth=1)
        ax.set_xlabel('Leverage')
        ax.set_ylabel('Standardized Residuals')
        ax.set_title("Residuals vs Leverage")

        # Cook's distance contours
        x_range = np.linspace(0.001, ax.get_xlim()[1], 100)
        for cook_val in [0.5, 1.0]:
            for sign in [1, -1]:
                p = model_result.df_model + 1
                y_val = sign * np.sqrt(cook_val * p * (1 - x_range) / x_range)
                ax.plot(x_range, y_val, '--', color=COLORS['bias'],
                        alpha=0.5, label=f"Cook's d={cook_val}" if sign == 1 else None)
        ax.legend(fontsize=8)

        fig.tight_layout()
        return fig

    @staticmethod
    def summary_tests(model_result):
        """
        Return dict of diagnostic test results:
        vif, breusch_pagan, durbin_watson, jarque_bera.
        """
        results = {}

        # VIF — requires exog with constant
        exog = model_result.model.exog
        exog_names = model_result.model.exog_names
        vif_dict = {}
        for i, name in enumerate(exog_names):
            if name == 'const':
                continue
            vif_dict[name] = variance_inflation_factor(exog, i)
        results['vif'] = vif_dict

        # Breusch-Pagan
        bp_stat, bp_pval, _, _ = het_breuschpagan(
            model_result.resid, model_result.model.exog
        )
        results['breusch_pagan'] = (bp_stat, bp_pval)

        # Durbin-Watson
        results['durbin_watson'] = durbin_watson(model_result.resid)

        # Jarque-Bera
        jb_stat, jb_pval, _, _ = jarque_bera(model_result.resid)
        results['jarque_bera'] = (jb_stat, jb_pval)

        return results


# ---------------------------------------------------------------------------
# AutopsyReport — widget factory for structured reflection
# ---------------------------------------------------------------------------
class AutopsyReport:
    """Creates reflection widgets for notebook wrap-up cells."""

    @staticmethod
    def lightweight(notebook_number):
        """Two-question mini autopsy for Notebooks 3-6."""
        threat = widgets.Text(
            description='Biggest threat:',
            placeholder='What is the biggest threat to this estimate?',
            layout=widgets.Layout(width='90%'),
            style={'description_width': '120px'},
        )
        check = widgets.Text(
            description='How to check:',
            placeholder='How would you check if that threat is real?',
            layout=widgets.Layout(width='90%'),
            style={'description_width': '120px'},
        )
        submit = widgets.Button(description='Save Autopsy', button_style='primary')
        output = widgets.Output()

        def on_submit(btn):
            with output:
                output.clear_output()
                nb_id = f"notebook_{notebook_number}"
                record_trap_response(nb_id, "threat", threat.value)
                record_trap_response(nb_id, "check", check.value)
                print("Autopsy responses saved.")
                submit.disabled = True

        submit.on_click(on_submit)
        return widgets.VBox([
            widgets.HTML(f"<h3>Mini Autopsy — Notebook {notebook_number}</h3>"),
            threat, check, submit, output
        ])

    @staticmethod
    def full(notebook_number, available_sidebars=None):
        """Full sensitivity-analysis autopsy for Notebooks 7-8."""
        fields = {
            'point_estimate': widgets.Text(
                description='Point estimate:',
                placeholder='My point estimate is:',
                layout=widgets.Layout(width='90%'),
                style={'description_width': '150px'},
            ),
            'robustness': widgets.Text(
                description='Robustness value:',
                placeholder='The robustness value is:',
                layout=widgets.Layout(width='90%'),
                style={'description_width': '150px'},
            ),
            'partial_r2': widgets.Text(
                description='Strongest partial R²:',
                placeholder='The strongest observed covariate has partial R² of:',
                layout=widgets.Layout(width='90%'),
                style={'description_width': '150px'},
            ),
            'plain_language': widgets.Text(
                description='Plain language:',
                placeholder='An omitted variable would need to be ___ times as important as ___ to explain away this result',
                layout=widgets.Layout(width='90%'),
                style={'description_width': '150px'},
            ),
        }

        children = [
            widgets.HTML(f"<h3>Full Autopsy Report — Notebook {notebook_number}</h3>"),
        ]
        children.extend(fields.values())

        if available_sidebars:
            sidebar_dropdown = widgets.Dropdown(
                options=['(select)'] + list(available_sidebars),
                description='Most analogous disaster:',
                layout=widgets.Layout(width='90%'),
                style={'description_width': '180px'},
            )
            children.append(sidebar_dropdown)
            fields['analogous_disaster'] = sidebar_dropdown

        submit = widgets.Button(description='Save Autopsy', button_style='primary')
        output = widgets.Output()

        def on_submit(btn):
            with output:
                output.clear_output()
                nb_id = f"notebook_{notebook_number}"
                for key, w in fields.items():
                    record_trap_response(nb_id, key, w.value)
                print("Full autopsy report saved.")
                submit.disabled = True

        submit.on_click(on_submit)
        children.extend([submit, output])
        return widgets.VBox(children)

# ── End embedded company_sim.py ──

from IPython.display import display, HTML, Markdown
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
%matplotlib inline

# Fast mode: set FAST_TEST=1 to reduce precomputation
FAST_TEST = os.environ.get('FAST_TEST', '0') == '1'

# ── NB3-specific DGP ──
# True model with variables of varying practical significance
NB3_BETAS = {
    'ad_spend': 8.0,        # $8 per unit — meaningful
    'staff_count': 3.0,     # $3 per unit — meaningful
    'satisfaction': 0.003,  # $0.003 per unit — trivial!
    'store_age': 47.2,     # $47.2 per year — very meaningful
    'wifi_speed': 0.001,   # $0.001 per mbps — trivial!
}

class NB3Simulator:
    """DGP with mix of meaningful and trivial effects."""
    def __init__(self, n=500):
        self.n = n

    def generate(self, n=None, seed=42):
        if n is None:
            n = self.n
        rng = np.random.default_rng(seed)
        ad_spend = rng.uniform(1, 20, n)
        staff_count = rng.standard_normal(n) * 2 + 5
        satisfaction = rng.uniform(1, 100, n)  # 1-100 scale
        store_age = rng.uniform(0.5, 10, n)    # years
        wifi_speed = rng.uniform(10, 500, n)    # mbps
        eps = rng.standard_normal(n) * 2.0
        Y = (50 + NB3_BETAS["ad_spend"] * ad_spend
             + NB3_BETAS["staff_count"] * staff_count
             + NB3_BETAS["satisfaction"] * satisfaction
             + NB3_BETAS["store_age"] * store_age
             + NB3_BETAS["wifi_speed"] * wifi_speed
             + eps)
        return pd.DataFrame({
            'revenue': Y, 'ad_spend': ad_spend,
            'staff_count': staff_count, 'satisfaction': satisfaction,
            'store_age': store_age, 'wifi_speed': wifi_speed,
        })

nb3_sim = NB3Simulator(n=50000)

# ── Precompute MC cache for sample size slider ──
print("Precomputing Monte Carlo simulations...")

n_grid_log = [50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000, 100000]
N_REPS = 50 if FAST_TEST else 2000

mc_cache_n = {}  # n -> array of (beta, se, pval) per variable per rep
progress = widgets.IntProgress(value=0, min=0, max=len(n_grid_log),
    description='Precomputing:', style={'description_width': 'initial'})
display(progress)

vars_list = ['ad_spend', 'staff_count', 'satisfaction', 'store_age', 'wifi_speed']

for i, n_val in enumerate(n_grid_log):
    cache_entry = {v: {"beta": [], "se": [], "pval": []} for v in vars_list}
    sim_temp = NB3Simulator(n=n_val)
    for r in range(N_REPS):
        data = sim_temp.generate(n=n_val, seed=r)
        Xd = sm.add_constant(data[vars_list])
        model = sm.OLS(data['revenue'], Xd).fit(cov_type='HC3')
        for v in vars_list:
            cache_entry[v]["beta"].append(model.params[v])
            cache_entry[v]["se"].append(model.bse[v])
            cache_entry[v]["pval"].append(model.pvalues[v])
    for v in vars_list:
        for k in ["beta", "se", "pval"]:
            cache_entry[v][k] = np.array(cache_entry[v][k])
    mc_cache_n[n_val] = cache_entry
    progress.value = i + 1

progress.bar_style = 'success'

# ── Power analysis cache ──
# For varying true effect size, compute power at fixed n=500
effect_grid = np.round(np.arange(0.0, 5.1, 0.25), 2)
mc_cache_power = {}
progress2 = widgets.IntProgress(value=0, min=0, max=len(effect_grid),
    description='Power cache:', style={'description_width': 'initial'})
display(progress2)

for i, eff in enumerate(effect_grid):
    rejections = np.empty(N_REPS)
    for r in range(N_REPS):
        rng = np.random.default_rng(r)
        n = 500
        X = rng.standard_normal(n)
        Y = eff * X + rng.standard_normal(n)
        data = pd.DataFrame({'Y': Y, 'X': X})
        Xd = sm.add_constant(data['X'])
        model = sm.OLS(data['Y'], Xd).fit()
        rejections[r] = float(model.pvalues['X'] < 0.05)
    mc_cache_power[round(eff, 2)] = rejections
    progress2.value = i + 1

progress2.bar_style = 'success'
print("\nSetup complete!")


# Notebook 3: Why Your Significance Is Wrong

---

## The NovaMart Scenario (continued)

Your regression is solid: correct controls (Notebook 1), robust standard errors (Notebook 2). NovaMart's data team has collected a massive dataset — **50,000 store observations**. With this much data, your results should be rock-solid.

You run the regression and every variable comes back as "highly significant" (p < 0.001). The board is thrilled: *"Everything matters! Let's invest in all five factors."*

In [ ]:
# ── The Trap: 50,000 observations, everything is "significant" ──
data = nb3_sim.generate(n=50000, seed=42)

vars_list = ['ad_spend', 'staff_count', 'satisfaction', 'store_age', 'wifi_speed']
X = sm.add_constant(data[vars_list])
model = sm.OLS(data['revenue'], X).fit(cov_type='HC3')
print(model.summary())

print("\n" + "="*60)
print("  ALL variables have p < 0.001!")
print("="*60)

# ── Trap Widget ──
trap = create_trap_widget(
    notebook_id="notebook_3",
    question_id="q1",
    question_text=(
        "Based on the p-values, which variables have a MEANINGFUL effect on revenue? "
        "Select the best answer."
    ),
    options=[
        'A: All of them — every p-value is below 0.001.',
        'B: Only ad_spend and store_age — they have the largest coefficients.',
        'C: Cannot determine from p-values alone — need effect sizes.',
        'D: None — with 50K observations, significance is meaningless.',
    ]
)
display(trap)


In [ ]:
# ── The Reveal ──
if not check_gate("notebook_3", "q1"):
    display(HTML('<div style="background:#FFCCCC;padding:15px;border-radius:8px;">'
                 '<b>Please answer the question in Cell 2 before continuing.</b></div>'))
else:
    response = get_trap_response("notebook_3", "q1")
    print(f"Your answer: {response}\n")

    if response.startswith("A"):
        display(HTML('<div style="background:#FFDDDD;padding:12px;border-radius:8px;">'
            '<b>Classic trap!</b> Statistical significance ≠ practical significance. '
            'With enough data, even a $0.003 effect becomes "significant."</div>'))
    elif response.startswith("B"):
        display(HTML('<div style="background:#D4EDDA;padding:12px;border-radius:8px;">'
            '<b>Good instinct!</b> You looked at effect sizes, not just p-values. '
            "Let's formalize why this matters.</div>"))
    elif response.startswith("C"):
        display(HTML('<div style="background:#D4EDDA;padding:12px;border-radius:8px;">'
            '<b>Correct!</b> P-values alone cannot tell you if an effect is meaningful. '
            "Let's see why.</div>"))
    elif response.startswith("D"):
        display(HTML('<div style="background:#FFF3CD;padding:12px;border-radius:8px;">'
            '<b>Too extreme.</b> Significance is not meaningless — but it is not sufficient. '
            'Some of these effects ARE meaningful AND significant.</div>'))

    # ── Killer visualization: coefficient vs practical significance ──
    data = nb3_sim.generate(n=50000, seed=42)
    vars_list = ['ad_spend', 'staff_count', 'satisfaction', 'store_age', 'wifi_speed']
    X = sm.add_constant(data[vars_list])
    model = sm.OLS(data['revenue'], X).fit(cov_type='HC3')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: coefficients with CIs
    ax = axes[0]
    coefs = [model.params[v] for v in vars_list]
    ci_low = [model.conf_int().loc[v, 0] for v in vars_list]
    ci_high = [model.conf_int().loc[v, 1] for v in vars_list]
    colors = [COLORS['repair'] if abs(c) > 1.0 else COLORS['bias'] for c in coefs]

    y_pos = np.arange(len(vars_list))
    ax.barh(y_pos, coefs, color=colors, alpha=0.7, edgecolor='white')
    for i, (lo, hi) in enumerate(zip(ci_low, ci_high)):
        ax.plot([lo, hi], [i, i], color='black', linewidth=1.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(vars_list)
    ax.set_xlabel('Coefficient ($ per unit)', fontsize=11)
    ax.set_title('Coefficient Size', fontsize=12, fontweight='bold')
    ax.axvline(0, color='gray', linewidth=0.5)

    # Right: comparison table
    ax = axes[1]
    ax.axis('off')
    table_data = []
    for v in vars_list:
        coef = model.params[v]
        se = model.bse[v]
        pval = model.pvalues[v]
        practical = "MEANINGFUL" if abs(coef) > 1.0 else "TRIVIAL"
        table_data.append([v, f"${coef:.3f}", f"${se:.4f}", f"{pval:.1e}", practical])

    table = ax.table(
        cellText=table_data,
        colLabels=['Variable', 'Coefficient', 'Std Error', 'p-value', 'Practical?'],
        loc='center', cellLoc='center',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.8)

    # Color the practical significance column
    for i, row in enumerate(table_data):
        cell = table[i + 1, 4]
        if row[4] == "TRIVIAL":
            cell.set_facecolor('#FFDDDD')
        else:
            cell.set_facecolor('#D4EDDA')

    ax.set_title('Statistical vs Practical Significance', fontsize=12, fontweight='bold')

    fig.tight_layout()
    plt.show()

    display(HTML(
        '<div style="background:#F0F4FF;padding:20px;border-radius:10px;'
        'border-left:4px solid #2E5EA8;margin-top:15px;">'
        '<h3>The Significance Decomposition</h3>'
        r'<p style="font-size:16px;">$$t = \frac{\hat{\beta}}{SE(\hat{\beta})}$$</p>'
        r'<p style="font-size:16px;">$$\text{Power} = P(|t| > c \mid \beta \neq 0)$$</p>'
        '<p>As n grows:</p><ul>'
        r'<li>$\hat{\beta}$ stays the same (unbiased)</li>'
        r'<li>$SE(\hat{\beta})$ shrinks as $1/\sqrt{n}$</li>'
        r'<li>$t$ grows, so $p$-value shrinks</li>'
        '</ul>'
        '<p><b>With enough data, ANY non-zero effect — no matter how tiny — becomes '
        '"statistically significant." The p-value measures evidence against zero, '
        'not evidence of importance.</b></p>'
        '</div>'
    ))


In [ ]:
# ── Monte Carlo Confirmation ──
# Show how SE shrinks and p-value plummets for a TRIVIAL effect (satisfaction)
trivial_var = 'satisfaction'
true_coef = NB3_BETAS[trivial_var]

n_keys = sorted(mc_cache_n.keys())
mean_betas = [mc_cache_n[n][trivial_var]['beta'].mean() for n in n_keys]
mean_ses = [mc_cache_n[n][trivial_var]['se'].mean() for n in n_keys]
mean_pvals = [mc_cache_n[n][trivial_var]['pval'].mean() for n in n_keys]
frac_sig = [(mc_cache_n[n][trivial_var]['pval'] < 0.05).mean() * 100 for n in n_keys]

fig_mc, axes = plt.subplots(1, 3, figsize=(15, 4.5))

skip_btn = widgets.Button(description='Skip Animation', button_style='warning')
skip_flag = [False]
def on_skip(btn):
    skip_flag[0] = True
    btn.disabled = True
skip_btn.on_click(on_skip)
display(skip_btn)

n_frames = min(200, len(n_keys) * 20)
for frame in range(n_frames):
    if skip_flag[0]:
        break
    show_idx = min(frame // 20 + 1, len(n_keys))

    for ax in axes:
        ax.clear()

    # Coefficient (flat)
    axes[0].plot(n_keys[:show_idx], mean_betas[:show_idx],
                 color=COLORS['ols'], linewidth=2, marker='o', markersize=5)
    axes[0].axhline(true_coef, color=COLORS['truth'], linewidth=2, linestyle='--',
                    label=f'True = {true_coef}')
    axes[0].set_xlabel('Sample Size')
    axes[0].set_ylabel('Mean Coefficient')
    axes[0].set_title(f'{trivial_var}: Coefficient (stable)', fontsize=11)
    axes[0].set_xscale('log')
    axes[0].legend(fontsize=9)

    # SE (shrinking)
    axes[1].plot(n_keys[:show_idx], mean_ses[:show_idx],
                 color=COLORS['ols'], linewidth=2, marker='o', markersize=5)
    axes[1].set_xlabel('Sample Size')
    axes[1].set_ylabel('Mean Standard Error')
    axes[1].set_title(f'{trivial_var}: SE (shrinking)', fontsize=11)
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')

    # p-value / fraction significant
    axes[2].plot(n_keys[:show_idx], frac_sig[:show_idx],
                 color=COLORS['bias'], linewidth=2, marker='o', markersize=5)
    axes[2].axhline(5, color='gray', linewidth=1, linestyle=':', alpha=0.5)
    axes[2].set_xlabel('Sample Size')
    axes[2].set_ylabel('% Rejections (p < 0.05)')
    axes[2].set_title(f'{trivial_var}: "Significant" (plummeting p)', fontsize=11)
    axes[2].set_xscale('log')
    axes[2].set_ylim(0, 105)

    fig_mc.tight_layout()
    fig_mc.canvas.draw()
    fig_mc.canvas.flush_events()

# Final
for ax in axes:
    ax.clear()

axes[0].plot(n_keys, mean_betas, color=COLORS['ols'], linewidth=2, marker='o', markersize=5)
axes[0].axhline(true_coef, color=COLORS['truth'], linewidth=2, linestyle='--',
                label=f'True = {true_coef}')
axes[0].set_xlabel('Sample Size')
axes[0].set_ylabel('Mean Coefficient')
axes[0].set_title(f'{trivial_var}: Coefficient stays at ${true_coef}', fontsize=11, fontweight='bold')
axes[0].set_xscale('log')
axes[0].legend(fontsize=9)

axes[1].plot(n_keys, mean_ses, color=COLORS['ols'], linewidth=2, marker='o', markersize=5)
axes[1].set_xlabel('Sample Size')
axes[1].set_ylabel('Mean Standard Error')
axes[1].set_title(f'{trivial_var}: SE shrinks with n', fontsize=11, fontweight='bold')
axes[1].set_xscale('log')
axes[1].set_yscale('log')

axes[2].plot(n_keys, frac_sig, color=COLORS['bias'], linewidth=2, marker='o', markersize=5)
axes[2].axhline(5, color='gray', linewidth=1, linestyle=':', alpha=0.5)
axes[2].set_xlabel('Sample Size')
axes[2].set_ylabel('% Rejections (p < 0.05)')
axes[2].set_title(f'{trivial_var}: Trivial effect crosses p<0.05 threshold', fontsize=11, fontweight='bold')
axes[2].set_xscale('log')
axes[2].set_ylim(0, 105)

fig_mc.tight_layout()
plt.show()

print(f"\n  Satisfaction coefficient: ${true_coef} per unit")
print(f"  At n=50,000: {frac_sig[n_keys.index(50000)]:.0f}% of regressions reject H0")
print(f"  But the effect is $0.003 — completely meaningless in practice!")


---
## Discussion Prompt

> **Paper A reports p = 0.048. Paper B reports p = 0.052. Your colleague says "A found an effect, B didn't." Do you agree?**

Think about:
- Is the difference between p = 0.048 and p = 0.052 meaningful?
- What if Paper A had n = 10,000 and Paper B had n = 50?
- Should the 0.05 threshold be treated as a bright line?

In [ ]:
# ── The Destruction Playground: Sample size slider ──
fig_dest, axes_dest = plt.subplots(1, 3, figsize=(16, 5))
plt.subplots_adjust(wspace=0.35)

n_slider = widgets.SelectionSlider(
    options=[50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000, 100000],
    value=500,
    description='Sample size n:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='500px'),
)

vars_display = ['ad_spend', 'satisfaction', 'store_age', 'wifi_speed']
var_colors = {
    'ad_spend': COLORS['repair'],
    'satisfaction': COLORS['bias'],
    'store_age': COLORS['truth'],
    'wifi_speed': '#7F8C8D',
}

def update_destruction(change):
    n_val = change['new'] if isinstance(change, dict) else change

    for ax in axes_dest:
        ax.clear()

    n_keys = sorted(mc_cache_n.keys())
    up_to = [k for k in n_keys if k <= n_val]

    # Panel 1: Coefficient (flat)
    for v in vars_display:
        means = [mc_cache_n[n][v]['beta'].mean() for n in up_to]
        axes_dest[0].plot(up_to, means, linewidth=2, marker='o', markersize=3,
                          color=var_colors[v], label=f'{v} (true={NB3_BETAS[v]})')
    axes_dest[0].set_xlabel('Sample Size')
    axes_dest[0].set_ylabel('Mean Coefficient')
    axes_dest[0].set_title(f'Coefficients (n={n_val:,})', fontsize=11, fontweight='bold')
    axes_dest[0].set_xscale('log')
    axes_dest[0].legend(fontsize=8)

    # Panel 2: SE (shrinking)
    for v in vars_display:
        ses = [mc_cache_n[n][v]['se'].mean() for n in up_to]
        axes_dest[1].plot(up_to, ses, linewidth=2, marker='o', markersize=3,
                          color=var_colors[v], label=v)
    axes_dest[1].set_xlabel('Sample Size')
    axes_dest[1].set_ylabel('Mean Standard Error')
    axes_dest[1].set_title(f'Standard Errors (shrinking)', fontsize=11, fontweight='bold')
    axes_dest[1].set_xscale('log')
    axes_dest[1].set_yscale('log')
    axes_dest[1].legend(fontsize=8)

    # Panel 3: % significant
    for v in vars_display:
        fracs = [(mc_cache_n[n][v]['pval'] < 0.05).mean() * 100 for n in up_to]
        axes_dest[2].plot(up_to, fracs, linewidth=2, marker='o', markersize=3,
                          color=var_colors[v], label=v)
    axes_dest[2].axhline(5, color='gray', linewidth=1, linestyle=':', alpha=0.5)
    axes_dest[2].axhline(80, color='gray', linewidth=1, linestyle='--', alpha=0.3,
                          label='80% power line')
    axes_dest[2].set_xlabel('Sample Size')
    axes_dest[2].set_ylabel('% Significant (p < 0.05)')
    axes_dest[2].set_title(f'Significance Rate', fontsize=11, fontweight='bold')
    axes_dest[2].set_xscale('log')
    axes_dest[2].set_ylim(0, 105)
    axes_dest[2].legend(fontsize=8)

    fig_dest.canvas.draw_idle()

n_slider.observe(update_destruction, names='value')
update_destruction({'new': 500})
display(n_slider)
plt.show()


In [ ]:
# ── The Repair: Effect size & Power analysis ──
fig_rep, (ax_eff, ax_pow) = plt.subplots(1, 2, figsize=(14, 5))
plt.subplots_adjust(wspace=0.35)

toggle_effect = widgets.ToggleButton(value=False,
    description='Show Effect Size Measures',
    button_style='', layout=widgets.Layout(width='250px'))

effect_slider = widgets.SelectionSlider(
    options=[round(v, 2) for v in np.arange(0.0, 5.1, 0.25)],
    value=1.0, description='True effect:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px'))

def update_repair(change=None):
    ax_eff.clear()
    ax_pow.clear()

    if toggle_effect.value:
        toggle_effect.button_style = 'success'

        # Left: Effect sizes for our NovaMart variables
        data = nb3_sim.generate(n=50000, seed=42)
        vars_list = ['ad_spend', 'staff_count', 'satisfaction', 'store_age', 'wifi_speed']
        X = sm.add_constant(data[vars_list])
        model = sm.OLS(data['revenue'], X).fit(cov_type='HC3')

        # Compute standardized effect sizes (Cohen's f2-like)
        coefs = [model.params[v] for v in vars_list]
        # Dollar impact per 1 SD change
        sd_impacts = [model.params[v] * data[v].std() for v in vars_list]

        y_pos = np.arange(len(vars_list))
        colors = [COLORS['repair'] if abs(s) > 5 else COLORS['bias'] for s in sd_impacts]
        ax_eff.barh(y_pos, sd_impacts, color=colors, alpha=0.8, edgecolor='white')
        ax_eff.set_yticks(y_pos)
        ax_eff.set_yticklabels(vars_list)
        ax_eff.set_xlabel('Revenue impact per 1 SD change ($)', fontsize=11)
        ax_eff.set_title('Practical Effect Size (standardized)', fontsize=12, fontweight='bold')
        ax_eff.axvline(0, color='gray', linewidth=0.5)

        for i, s in enumerate(sd_impacts):
            label = f'${s:.1f}'
            ax_eff.text(s + (1 if s > 0 else -1), i, label, va='center', fontsize=10)
    else:
        toggle_effect.button_style = ''
        ax_eff.text(0.5, 0.5, 'Toggle "Show Effect Size"\nto activate',
                    ha='center', va='center', fontsize=12, color='gray',
                    transform=ax_eff.transAxes)
        ax_eff.set_title('Effect Size Measures', fontsize=12, color='gray')

    # Right: Power analysis curve
    eff_val = round(effect_slider.value, 2)
    eff_keys = sorted(mc_cache_power.keys())
    powers = [mc_cache_power[k].mean() * 100 for k in eff_keys]

    ax_pow.plot(eff_keys, powers, color=COLORS['ols'], linewidth=2, marker='o', markersize=4)
    ax_pow.axhline(80, color=COLORS['truth'], linewidth=1.5, linestyle='--',
                   label='80% power threshold')
    ax_pow.axhline(5, color='gray', linewidth=1, linestyle=':', alpha=0.5,
                   label='Type I error rate')
    ax_pow.axvline(eff_val, color=COLORS['bias'], linewidth=1.5, linestyle=':', alpha=0.7)

    current_power = mc_cache_power[eff_val].mean() * 100
    ax_pow.set_xlabel('True Effect Size', fontsize=11)
    ax_pow.set_ylabel('Power (%)', fontsize=11)
    ax_pow.set_title(f'Power at n=500: {current_power:.0f}% for effect={eff_val}',
                     fontsize=12, fontweight='bold')
    ax_pow.set_ylim(0, 105)
    ax_pow.legend(fontsize=9)

    fig_rep.canvas.draw_idle()

toggle_effect.observe(update_repair, names='value')
effect_slider.observe(update_repair, names='value')
update_repair()
display(widgets.HBox([toggle_effect]))
display(effect_slider)
plt.show()


In [ ]:
# ── The Limit: Type II errors with small n ──
fig_lim, (ax_power, ax_summ) = plt.subplots(1, 2, figsize=(13, 5),
    gridspec_kw={'width_ratios': [2, 1]})
plt.subplots_adjust(wspace=0.35)

effect_lim_slider = widgets.SelectionSlider(
    options=[round(v, 2) for v in np.arange(0.0, 5.1, 0.25)],
    value=2.0, description='True effect:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px'))

def update_limit(change=None):
    eff_val = round(effect_lim_slider.value, 2)

    ax_power.clear()
    eff_keys = sorted(mc_cache_power.keys())
    powers = [mc_cache_power[k].mean() * 100 for k in eff_keys]
    type2_rates = [100 - p for p in powers]

    ax_power.plot(eff_keys, type2_rates, color=COLORS['bias'], linewidth=2,
                  marker='o', markersize=4, label='Type II Error Rate')
    ax_power.axhline(20, color=COLORS['truth'], linewidth=1.5, linestyle='--',
                     label='20% Type II (80% power)')
    ax_power.axvline(eff_val, color=COLORS['ols'], linewidth=1.5, linestyle=':', alpha=0.7)
    ax_power.set_xlabel('True Effect Size', fontsize=11)
    ax_power.set_ylabel('Type II Error Rate (%)', fontsize=11)

    current_power = mc_cache_power[eff_val].mean() * 100
    type2 = 100 - current_power
    status = "Repair effective" if current_power >= 80 else "Repair unreliable"
    title_color = COLORS['repair'] if "effective" in status else COLORS['bias']
    ax_power.set_title(f'{status} (effect = {eff_val}, n=500)',
                       fontsize=12, fontweight='bold', color=title_color)
    ax_power.set_ylim(0, 100)
    ax_power.legend(fontsize=9)

    ax_summ.clear()
    ax_summ.axis('off')
    summary_text = (
        f"True effect = {eff_val}\n"
        f"Sample size = 500\n\n"
        f"Power: {current_power:.1f}%\n"
        f"Type II rate: {type2:.1f}%\n\n"
        f"{'Large true effects are\ndetectable' if current_power >= 80 else 'TRUE EFFECT MISSED!\nInsignificant does NOT\nmean no effect'}"
    )
    ax_summ.text(0.1, 0.5, summary_text, fontsize=13,
                 verticalalignment='center', fontfamily='monospace',
                 bbox=dict(boxstyle='round,pad=0.8', facecolor='#F8F8F8',
                           edgecolor=title_color, linewidth=2))
    ax_summ.set_title('Summary', fontsize=12, fontweight='bold')

    fig_lim.canvas.draw_idle()

effect_lim_slider.observe(update_limit, names='value')
update_limit()
display(effect_lim_slider)
plt.show()


In [ ]:
#@title Real-World Disaster: GWAS and the Multiple Testing Catastrophe

story_html = (
    '<div style="padding:15px;font-size:14px;line-height:1.6;">'
    '<h4>Genome-Wide Association Studies: 50,000 False Positives</h4>'
    '<p>In a typical GWAS, researchers test <b>1 million</b> genetic variants for association '
    'with a disease. Using the standard p < 0.05 threshold, you expect '
    '<b>50,000 false positives</b> even if no variant has any real effect.</p>'
    '<p>This led to the "missing heritability" problem and the adoption of the Bonferroni '
    'correction: p < 0.05/1,000,000 = 5e-8. Most "significant" findings from early GWAS '
    'studies failed to replicate.</p>'
    '</div>'
)

math_html = (
    '<div style="padding:15px;font-size:14px;line-height:1.8;">'
    '<h4>The Multiple Testing Problem</h4>'
    r'<p>With $m$ independent tests at level $\alpha$:</p>'
    r'<p>$$\text{Expected false positives} = m \cdot \alpha$$</p>'
    r'<p>For $m = 1,000,000$ and $\alpha = 0.05$:</p>'
    r'<p>$$\text{Expected false positives} = 1,000,000 \times 0.05 = 50,000$$</p>'
    r'<p><b>Bonferroni correction:</b> $\alpha_{\text{adj}} = \alpha / m = 5 \times 10^{-8}$</p>'
    '<p>This is extremely conservative but necessary when testing millions of hypotheses.</p>'
    '</div>'
)

sim_output = widgets.Output()
sim_btn = widgets.Button(description='Run GWAS Simulation', button_style='primary')

def run_gwas_sim(btn):
    with sim_output:
        sim_output.clear_output(wait=True)
        print("Simulating 1M t-tests on pure noise...")
        rng = np.random.default_rng(42)
        n_tests = 1_000_000
        n_obs = 100

        # Generate t-statistics for null (no real effect)
        t_stats = rng.standard_t(df=n_obs-2, size=n_tests)
        from scipy.stats import t as t_dist
        p_values = 2 * (1 - t_dist.cdf(np.abs(t_stats), df=n_obs-2))

        false_positives_005 = (p_values < 0.05).sum()
        false_positives_bonf = (p_values < 0.05 / n_tests).sum()

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

        # Left: histogram of p-values
        ax1.hist(p_values, bins=100, density=True,
                 color=COLORS['ols'], alpha=0.7, edgecolor='white')
        ax1.axhline(1.0, color=COLORS['truth'], linewidth=2, linestyle='--',
                    label='Uniform (expected under null)')
        ax1.set_xlabel('p-value')
        ax1.set_ylabel('Density')
        ax1.set_title('1M p-values Under the Null', fontweight='bold')
        ax1.legend()

        # Right: false positive counts
        ax2.bar(['p < 0.05', 'Bonferroni\np < 5e-8'],
                [false_positives_005, false_positives_bonf],
                color=[COLORS['bias'], COLORS['repair']], alpha=0.8)
        ax2.set_ylabel('Number of "Significant" Results')
        ax2.set_title('False Positives from 1M Null Tests', fontweight='bold')
        for i, v in enumerate([false_positives_005, false_positives_bonf]):
            ax2.text(i, v + 500, f'{v:,}', ha='center', fontsize=12, fontweight='bold')

        fig.tight_layout()
        plt.show()

        print(f"\n  Tests run: {n_tests:,}")
        print(f"  False positives at p < 0.05:     {false_positives_005:,}")
        print(f"  False positives at Bonferroni:    {false_positives_bonf:,}")
        print(f"  True effects: 0 (all null!)")

sim_btn.on_click(run_gwas_sim)

tab = widgets.Tab(children=[
    widgets.HTML(story_html),
    widgets.HTML(math_html),
    widgets.VBox([sim_btn, sim_output]),
])
tab.set_title(0, 'Story')
tab.set_title(1, 'Math')
tab.set_title(2, 'Simulation')
display(tab)


In [ ]:
# ── Mini Autopsy ──
autopsy = AutopsyReport.lightweight(3)
display(autopsy)


---
## What's Next?

You now know what's significant and what actually matters. You can distinguish a p-value from practical importance, and you understand why large samples make everything "significant."

**But is your model even the right shape?**

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 12px; color: white; margin-top: 20px;">
<h3 style="color: white; margin-top: 0;">Notebook 4: Why Your Model Is Wrong</h3>
<p>Your significance is fixed. Now let's break your functional form.</p>
<p><a href="04_functional_form.ipynb" style="color: #FFD700; font-weight: bold;">Open Notebook 4</a></p>
</div>